In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
# ============================================================
# Exact GEMM primitive:
#
#       [12 x 16] @ [16 x 16] -> [12 x 16]
#
# The explicit per-sample loop prevents PyTorch from logically
# flattening B x 12 into one larger matrix multiplication.
# ============================================================

def gemm_12x16_16x16(a, b):
    """
    a:
        [B, 12, 16]

    b:
        [16, 16]       shared matrix
        or
        [B, 16, 16]    per-sample matrix

    returns:
        [B, 12, 16]
    """
    assert a.ndim == 3
    assert a.shape[1:] == (12, 16)

    out = []

    if b.ndim == 2:
        assert b.shape == (16, 16)

        for i in range(a.shape[0]):
            # EXACT:
            # [12,16] @ [16,16]
            out.append(torch.mm(a[i], b))

    else:
        assert b.ndim == 3
        assert b.shape[0] == a.shape[0]
        assert b.shape[1:] == (16, 16)

        for i in range(a.shape[0]):
            # EXACT:
            # [12,16] @ [16,16]
            out.append(torch.mm(a[i], b[i]))

    return torch.stack(out, dim=0)


class GEMM16(nn.Module):
    """
    Trainable linear layer implemented exclusively as

        [12,16] @ [16,16]
    """

    def __init__(self):
        super().__init__()

        self.weight = nn.Parameter(torch.empty(16, 16))
        self.bias = nn.Parameter(torch.zeros(16))

        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        y = gemm_12x16_16x16(x, self.weight)
        return y + self.bias

In [ ]:
# ============================================================
# Attention
# ============================================================

class FixedShapeAttention(nn.Module):
    """
    Single-head self attention designed around the required GEMM.

    Q/K/V projections:
        12x16 @ 16x16

    K and V are padded from 12 to 16 tokens.

    Attention score:
        Q       = 12x16
        K_pad^T = 16x16

        12x16 @ 16x16 -> 12x16

    Attention/value:
        A     = 12x16
        V_pad = 16x16

        12x16 @ 16x16 -> 12x16

    The four padded keys are masked before softmax.
    """

    def __init__(self, dropout=0.0):
        super().__init__()

        self.q_proj = GEMM16()
        self.k_proj = GEMM16()
        self.v_proj = GEMM16()
        self.out_proj = GEMM16()

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # -----------------------------------------------
        # Each of these is:
        #
        # 12x16 @ 16x16 -> 12x16
        # -----------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        B = x.shape[0]

        # Add four dummy tokens:
        #
        # [12,16] -> [16,16]
        #
        zero_tokens = torch.zeros(
            B,
            4,
            16,
            dtype=x.dtype,
            device=x.device,
        )

        k16 = torch.cat((k, zero_tokens), dim=1)
        v16 = torch.cat((v, zero_tokens), dim=1)

        # ------------------------------------------------
        # Attention scores
        #
        # q            = [B,12,16]
        # k16.transpose = [B,16,16]
        #
        # EXACT GEMM:
        #
        # 12x16 @ 16x16 -> 12x16
        # ------------------------------------------------

        kt = k16.transpose(1, 2)

        scores = gemm_12x16_16x16(q, kt)
        scores = scores / math.sqrt(16.0)

        # Ignore the four dummy keys.
        mask = torch.zeros(
            1, 12, 16,
            dtype=torch.bool,
            device=x.device
        )

        mask[:, :, 12:] = True

        scores = scores.masked_fill(mask, float("-inf"))

        attention = torch.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        # ------------------------------------------------
        # Attention @ V
        #
        # attention = [B,12,16]
        # v16       = [B,16,16]
        #
        # EXACT GEMM:
        #
        # 12x16 @ 16x16 -> 12x16
        # ------------------------------------------------

        y = gemm_12x16_16x16(attention, v16)

        # Another 12x16 @ 16x16.
        y = self.out_proj(y)

        return y


# ============================================================
# Transformer block
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, dropout=0.1):
        super().__init__()

        self.norm1 = nn.LayerNorm(16)
        self.attention = FixedShapeAttention(dropout)

        self.norm2 = nn.LayerNorm(16)

        # Normally a ViT uses D -> 4D -> D.
        #
        # Here we deliberately use
        #
        #       16 -> 16 -> 16
        #
        # so BOTH MLP layers are the required GEMM.
        self.fc1 = GEMM16()
        self.fc2 = GEMM16()

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Attention residual
        x = x + self.attention(self.norm1(x))

        # MLP
        y = self.norm2(x)

        # 12x16 @ 16x16
        y = self.fc1(y)

        y = F.gelu(y)
        y = self.dropout(y)

        # 12x16 @ 16x16
        y = self.fc2(y)

        return x + y



In [ ]:
# ============================================================
# MNIST Vision Transformer
# ============================================================

class MNISTViT(nn.Module):

    def __init__(
        self,
        depth=4,
        dropout=0.1,
    ):
        super().__init__()

        # --------------------------------------------------
        # MNIST:
        #       28 x 28
        #
        # Pad one pixel left + right:
        #       28 x 30
        #
        # Patch size:
        #       7 x 10
        #
        # Grid:
        #       4 x 3
        #
        # Number of tokens:
        #       4 * 3 = 12
        #
        # Each token has D=16.
        # --------------------------------------------------

        self.patch_embed = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=(7, 10),
            stride=(7, 10),
        )

        self.position = nn.Parameter(
            torch.zeros(1, 12, 16)
        )

        nn.init.trunc_normal_(
            self.position,
            std=0.02
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(dropout)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(16)

        # Instead of a 16x10 classifier, use another
        # 12x16 @ 16x16 operation and reserve the first
        # ten output dimensions for MNIST classes.
        self.class_proj = GEMM16()

    def forward(self, x):

        # MNIST:
        # [B,1,28,28]

        # -> [B,1,28,30]
        x = F.pad(
            x,
            (1, 1, 0, 0)
        )

        # -> [B,16,4,3]
        x = self.patch_embed(x)

        assert x.shape[-2:] == (4, 3)

        # -> [B,16,12]
        x = x.flatten(2)

        # -> [B,12,16]
        x = x.transpose(1, 2)

        assert x.shape[1:] == (12, 16)

        x = x + self.position

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # Exact:
        # 12x16 @ 16x16
        x = self.class_proj(x)

        # Mean pooling over 12 tokens.
        #
        # [B,12,16] -> [B,16]
        x = x.mean(dim=1)

        # First 10 channels = digit logits.
        logits = x[:, :10]

        return logits

In [ ]:
# ============================================================
# Training
# ============================================================

def train(model, epochs=3):

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            (0.1307,),
            (0.3081,)
        ),
    ])

    train_dataset = datasets.MNIST(
        "./data",
        train=True,
        download=True,
        transform=transform,
    )

    test_dataset = datasets.MNIST(
        "./data",
        train=False,
        download=True,
        transform=transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True,
        num_workers=2,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=2,
    )



    print(
        "parameters:",
        sum(p.numel() for p in model.parameters())
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-3,
        weight_decay=1e-4,
    )

    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):

        # -----------------------------
        # Train
        # -----------------------------

        model.train()

        total_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # -----------------------------
        # Evaluate
        # -----------------------------

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in test_loader:

                images = images.to(device)
                labels = labels.to(device)

                logits = model(images)

                predictions = logits.argmax(dim=1)

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.numel()

        accuracy = (
            100.0 * correct / total
        )

        print(
            f"epoch={epoch + 1:02d} "
            f"loss={total_loss / len(train_loader):.4f} "
            f"accuracy={accuracy:.2f}%"
        )

In [ ]:
# the model
device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

print("device:", device)
model = MNISTViT(
        depth=4,
        dropout=0.1,
    ).to(device)

In [ ]:
train(model)

torch.save(
    model.state_dict(),
    "mnist_vit.pth"
)

print("Saved PyTorch model to mnist_vit.pth")

In [ ]:
model.eval()
model = model.cpu()

dummy_input = torch.randn(
    1, 1, 28, 28,
    dtype=torch.float32
)

with torch.no_grad():
    torch.onnx.export(
        model,
        dummy_input,
        "mnist_vit.onnx",

        # Important for this ONNX-MLIR flow
        dynamo=False,

        # Conservative ONNX version
        opset_version=13,

        # Fixed input/output names
        input_names=["image"],
        output_names=["logits"],

        # Put trained parameters into the model
        export_params=True,

        # Don't expose parameters as model inputs
        keep_initializers_as_inputs=False,

        # Useful with the legacy exporter
        do_constant_folding=True,
    )

print("Exported mnist_vit.onnx")

In [ ]:
import onnx

m = onnx.load("mnist_vit.onnx")

onnx.checker.check_model(m)

print("IR version:", m.ir_version)

for opset in m.opset_import:
    print(
        "opset:",
        opset.domain or "ai.onnx",
        opset.version
    )

print("\nOperators:")

ops = {}

for node in m.graph.node:
    ops[node.op_type] = ops.get(node.op_type, 0) + 1

for op, count in sorted(ops.items()):
    print(f"{op:20s} {count}")